In [1]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train"
OUT_TEST = "csv_test"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)

SR = 16000
WIN_DUR = 0.100   # 100 ms
HOP_DUR = 0.010   # 10 ms
FREQ_MAX = 2000

# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    """Return 1 if ANY wheeze overlaps window"""
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

def fill_short_zero_gaps(labels, max_gap=4):
    """
    Convert runs of 0s to 1s if they are between two 1s
    and length <= max_gap
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i  # first non-zero

            # Check bounded by ones
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1

    return labels

def enforce_min_wheeze_duration(labels, min_len=10):
    """
    Keep wheeze (1) only if it lasts for at least min_len frames.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i

            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1

    return labels

# =============================================================================
# FEATURE EXTRACTION (MATCHES FFT PLOT CODE)
# =============================================================================
def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []
    frame_labels = []

    for i, start in enumerate(range(0, len(audio) - win_len + 1, hop_len)):
        end = start + win_len
        frame = audio[start:end]

        frame_win = frame * np.hanning(len(frame))

        X = np.fft.rfft(frame_win)
        freqs = np.fft.rfftfreq(len(frame_win), d=1/SR)
        mag = np.abs(X)

        mag[0] = 0  # remove DC

        valid = freqs <= FREQ_MAX
        freqs = freqs[valid]
        mag = mag[valid]

        idx = np.argmax(mag)
        amp = mag[idx]
        freq = freqs[idx]

        t_start = start / SR
        t_end = end / SR
        label = wheeze_priority_label(labels, t_start, t_end)

        rows.append([
            Path(wav_path).name,
            t_start,
            amp,
            freq
        ])
        frame_labels.append(label)

    frame_labels = np.array(frame_labels)

    df = pd.DataFrame(rows, columns=[
        "file", "time_step_s", "amplitude", "frequency"
    ])
    df["label"] = frame_labels

    return df

# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []

for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []

for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)

# =============================================================================
# MODEL TRAINING
# =============================================================================
X_train = train_df[["amplitude", "frequency"]].values
y_train = train_df["label"].values

X_test = test_df[["amplitude", "frequency"]].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
model.fit(X_train_s, y_train)

# =============================================================================
# PREDICTION + SAVE TEST CSVs (PER FILE)
# =============================================================================
# Raw frame-wise predictions
test_df["predicted_label_raw"] = model.predict(X_test_s)
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

processed_preds = []

for fname, df_f in test_df.groupby("file"):
    preds = df_f["predicted_label_raw"].values

    # Fill short zero gaps (<= 40 ms)
    preds = fill_short_zero_gaps(preds, max_gap=4)
    
    # Enforce minimum wheeze duration (>= 100 ms)
    preds = enforce_min_wheeze_duration(preds, min_len=15)

    # (Optional but recommended)
    # Re-enforce duration to remove tiny runs created by gap fill
    # preds = enforce_min_wheeze_duration(preds, min_len=10)

    processed_preds.append(pd.Series(preds, index=df_f.index))

test_df["predicted_label"] = pd.concat(processed_preds).sort_index()

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE ")
print(f"Train CSVs → {OUT_TRAIN}/")
print(f"Test CSVs  → {OUT_TEST}/")

              precision    recall  f1-score   support

      Normal       0.66      0.61      0.63      9668
      Wheeze       0.82      0.85      0.84     20152

    accuracy                           0.77     29820
   macro avg       0.74      0.73      0.73     29820
weighted avg       0.77      0.77      0.77     29820

Confusion Matrix:
 [[ 5876  3792]
 [ 2980 17172]]

DONE 
Train CSVs → csv_train/
Test CSVs  → csv_test/


In [4]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import median_filter


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train_enhanced"
OUT_TEST = "csv_test_enhanced"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)
SMOOTH_WIN = 7   # frames (~70 ms)

SR = 16000
WIN_DUR = 0.100
HOP_DUR = 0.010
FREQ_MAX = 2000

# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

def enforce_min_wheeze_duration(labels, min_len=10):
    """
    Keep wheeze (1) only if it lasts at least min_len frames.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    """
    Convert short zero gaps between ones to ones.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels

def suppress_segments_without_low_peaks(preds, n_peaks, peak_thresh=3):
    """
    Remove wheeze segments where no frame has n_peaks < peak_thresh.
    """
    preds = preds.copy()
    n = len(preds)
    i = 0

    while i < n:
        if preds[i] == 1:
            start = i
            while i < n and preds[i] == 1:
                i += 1
            end = i

            # Check if ANY frame in the segment has n_peaks < peak_thresh
            if not np.any(n_peaks[start:end] < peak_thresh):
                preds[start:end] = 0
        else:
            i += 1

    return preds



# =============================================================================
# FFT FEATURE EXTRACTION (ENHANCED)
# =============================================================================
def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []

    for start in range(0, len(audio) - win_len + 1, hop_len):
        end = start + win_len
        frame = audio[start:end] * np.hanning(win_len)

        X = np.fft.rfft(frame)
        freqs = np.fft.rfftfreq(win_len, d=1/SR)
        mag = np.abs(X)
        mag[0] = 0

        valid = freqs <= FREQ_MAX
        freqs, mag = freqs[valid], mag[valid]

        # ---- Core FFT features ----
        idx = np.argmax(mag)
        amp = mag[idx]
        freq = freqs[idx]

        # ---- Spectral entropy ----
        p = mag / (np.sum(mag) + 1e-12)
        spec_entropy = -np.sum(p * np.log(p + 1e-12))

        # ---- Spectral bandwidth ----
        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-12)
        spec_bandwidth = np.sqrt(
            np.sum(((freqs - centroid) ** 2) * mag) / (np.sum(mag) + 1e-12)
        )

        # ---- Peak dominance ----
        peak_to_mean = amp / (np.mean(mag) + 1e-12)

        # ---- Number of significant peaks ----
        peaks, _ = find_peaks(mag, height=0.3 * amp)
        n_peaks = len(peaks)

        t_start = start / SR
        t_end = end / SR
        label = wheeze_priority_label(labels, t_start, t_end)

        rows.append([
            Path(wav_path).name,
            t_start,
            amp,
            freq,
            spec_entropy,
            spec_bandwidth,
            peak_to_mean,
            n_peaks,
            label
        ])

    return pd.DataFrame(rows, columns=[
        "file", "time_step_s",
        "amplitude", "frequency",
        "spec_entropy", "spec_bandwidth",
        "peak_to_mean", "n_peaks",
        "label"
    ])

# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []
for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []
for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)

# =============================================================================
# MODEL TRAINING
# =============================================================================
feature_cols = [
    "amplitude", "frequency",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean", "n_peaks"
]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values
# Flip labels: Normal=1, Wheeze=0
# y_train_flipped = 1 - y_train
# y_test_flipped = 1 - y_test

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

n_pos = np.sum(y_train == 1)   # Normal frames
n_neg = np.sum(y_train == 0)   # Wheeze frames

# scale_pos_weight = n_neg / n_pos
# print(f"scale_pos_weight (Normal) = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    # scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_s, y_train)

# =============================================================================
# PREDICTION + TEMPORAL SMOOTHING
# =============================================================================
# Probabilities correspond to flipped "Normal"
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

# Convert back to wheeze probability
# test_df["wheeze_prob"] = 1 - normal_prob

# ---- Temporal smoothing per file ----
test_df["wheeze_prob_smooth"] = (
    test_df.groupby("file")["wheeze_prob"]
    .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
)

# ---- Final decision ----
# ---- Initial frame-wise decision ----
test_df["predicted_label_raw"] = (
    test_df["wheeze_prob_smooth"] > 0.5
).astype(int)

# ---- Apply temporal constraints per file ----
final_preds = []

for fname, df_f in test_df.groupby("file"):
    preds = df_f["predicted_label_raw"].values
    peaks = df_f["n_peaks"].values

    # Fill short zero gaps (<= 40 ms)
    preds = fill_short_zero_gaps(preds, max_gap=5)
    
    # Minimum wheeze duration (>= 100 ms)
    preds = enforce_min_wheeze_duration(preds, min_len=15)

    # NEW BIAS: suppress segments without low-peak frames
    preds = suppress_segments_without_low_peaks(preds,peaks,peak_thresh=4)


    # Re-enforce duration (robustness)
    # preds = enforce_min_wheeze_duration(preds, min_len=10)

    final_preds.append(pd.Series(preds, index=df_f.index))

test_df["predicted_label"] = pd.concat(final_preds).sort_index()

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")


              precision    recall  f1-score   support

      Normal       0.61      0.74      0.67      9668
      Wheeze       0.86      0.77      0.81     20152

    accuracy                           0.76     29820
   macro avg       0.73      0.76      0.74     29820
weighted avg       0.78      0.76      0.77     29820

Confusion Matrix:
 [[ 7147  2521]
 [ 4601 15551]]

DONE
